# Measure LLM Personality with Psychology Inventories

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/02_measure_personality.ipynb)

PSYCTL doesn't just *steer* personalities — it **measures** them using real psychology instruments. This notebook demonstrates how to score an LLM on the [IPIP-NEO-120](https://ipip.ori.org/30FacetNEO-PI-RItems.htm), a validated 120-item Big Five personality inventory.

**In this notebook you will:**
1. Score a baseline LLM personality profile using token log-probabilities
2. Apply a steering vector and re-measure to see the shift
3. Visualize the comparison with a radar chart
4. Try other inventories (REI-40, SD4-28)

**Requirements:** Colab free tier (T4 GPU). [HuggingFace token](https://huggingface.co/settings/tokens) with access to [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).

**Time:** ~8 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_ID} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded | Memory: {model.get_memory_footprint() / 1e9:.1f} GB")

## IPIP-NEO-120: The Big Five

The [IPIP-NEO-120](https://ipip.ori.org/30FacetNEO-PI-RItems.htm) is a validated 120-item personality inventory measuring the Big Five:

- **N** (Neuroticism) — emotional instability, anxiety
- **E** (Extraversion) — sociability, assertiveness
- **O** (Openness) — curiosity, imagination
- **A** (Agreeableness) — cooperation, trust
- **C** (Conscientiousness) — discipline, organization

We score each item using the model's **token log-probabilities** (not chat output), giving a precise numeric personality profile.

In [ ]:
from psyctl.data.inventories import create_inventory

inventory = create_inventory("ipip_neo_120")
questions = inventory.get_questions()
print(f"Loaded {len(questions)} questions across {len(inventory.get_supported_traits())} Big Five domains")
print(f"Traits: {inventory.get_supported_traits()}")
print(f"\nSample question: \"{questions[0]['text']}\" (domain: {questions[0]['domain']})")

## Baseline Measurement

In [ ]:
from tqdm.auto import tqdm


def score_inventory(model_obj, tokenizer_obj, questions, inventory):
    """Score all inventory questions using token log-probabilities."""
    domain_scores = {}

    for q in tqdm(questions, desc="Scoring", ncols=80):
        choices = q.get("choices", [])
        if choices:
            sorted_choices = sorted(choices, key=lambda c: c["score"])
            min_s, max_s = sorted_choices[0]["score"], sorted_choices[-1]["score"]
            scale = "\n".join(f"{c['score']} = {c['text']}" for c in sorted_choices)
            prompt = (
                f'Rate how accurately this statement describes you:\n\n'
                f'Statement: "{q["text"]}"\n\n'
                f'Scale:\n{scale}\n\n'
                f'Your rating ({min_s}-{max_s}):'
            )
        else:
            min_s, max_s = 1, 5
            prompt = (
                f'Rate how accurately this statement describes you on a scale of 1 to 5:\n\n'
                f'Statement: "{q["text"]}"\n\n'
                f'Scale:\n1 = Very Inaccurate\n2 = Moderately Inaccurate\n'
                f'3 = Neither Accurate Nor Inaccurate\n4 = Moderately Accurate\n'
                f'5 = Very Accurate\n\nYour rating (1-5):'
            )

        inputs = tokenizer_obj(prompt, return_tensors="pt").to(model_obj.device)
        with torch.no_grad():
            logits = model_obj(**inputs).logits[0, -1, :]

        score_tokens = {}
        for s in range(min_s, max_s + 1):
            token_id = tokenizer_obj.encode(str(s), add_special_tokens=False)
            if token_id:
                score_tokens[s] = token_id[0]

        score_logits = torch.tensor([logits[tid].item() for tid in score_tokens.values()])
        probs = torch.softmax(score_logits, dim=0)
        score_values = torch.tensor(list(score_tokens.keys()), dtype=torch.float32)
        weighted_score = (probs * score_values).sum().item()

        if q["keyed"] == "minus":
            weighted_score = float(min_s + max_s) - weighted_score

        domain = q["domain"]
        if domain not in domain_scores:
            domain_scores[domain] = []
        domain_scores[domain].append(weighted_score)

    return inventory.calculate_scores(domain_scores)

In [ ]:
print("Scoring baseline personality profile...")
baseline_scores = score_inventory(model, tokenizer, questions, inventory)

print("\nBaseline Big Five Profile:")
print(f"{'Domain':<20} {'Raw':>8} {'Z-Score':>10} {'Percentile':>12}")
print("-" * 52)
for code in sorted(baseline_scores.keys()):
    s = baseline_scores[code]
    print(f"{s['domain_name']:<20} {s['raw_score']:>8.2f} {s['z_score']:>+10.2f} {s['percentile']:>11.1f}%")

## Steered Measurement

Now we apply the agreeableness steering vector and re-measure. We use `get_steering_applied_model()` which attaches hooks to the model so every forward pass is steered.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
from psyctl.core.steering_applier import SteeringApplier

VECTOR_REPO = "dalekwon/bipo-steering-vectors"
VECTOR_DIR = Path("./vectors")
VECTOR_DIR.mkdir(exist_ok=True)

vector_path = Path(hf_hub_download(
    repo_id=VECTOR_REPO,
    filename="bipo_steering_english_agreeableness.safetensors",
    local_dir=str(VECTOR_DIR),
    token=os.environ.get("HF_TOKEN"),
))

applier = SteeringApplier()

print("Applying agreeableness steering (strength=2.0) and scoring...")
steered_model, steered_tok = applier.get_steering_applied_model(
    model=model,
    tokenizer=tokenizer,
    steering_vector_path=vector_path,
    strength=2.0,
    prompt_length=0,
)

steered_scores = score_inventory(steered_model, steered_tok, questions, inventory)
steered_model.remove_steering()

print("\nBaseline vs Steered (Agreeableness +2.0):")
print(f"{'Domain':<20} {'Baseline':>10} {'Steered':>10} {'Change':>10} {'Pctl Base':>12} {'Pctl Steer':>12}")
print("-" * 76)
for code in sorted(baseline_scores.keys()):
    b = baseline_scores[code]
    s = steered_scores[code]
    change = s["raw_score"] - b["raw_score"]
    sign = "+" if change > 0 else ""
    print(
        f"{b['domain_name']:<20} {b['raw_score']:>10.2f} {s['raw_score']:>10.2f}"
        f" {sign}{change:>9.2f} {b['percentile']:>11.1f}% {s['percentile']:>11.1f}%"
    )

## Radar Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def radar_chart(baseline_scores, steered_scores, title="Big Five: Baseline vs Steered"):
    """Draw a radar chart comparing baseline and steered percentiles."""
    domains = sorted(baseline_scores.keys())
    labels = [baseline_scores[d]["domain_name"] for d in domains]
    baseline_vals = [baseline_scores[d]["percentile"] for d in domains]
    steered_vals = [steered_scores[d]["percentile"] for d in domains]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    baseline_vals += baseline_vals[:1]
    steered_vals += steered_vals[:1]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, baseline_vals, "o-", linewidth=2, label="Baseline", color="#4A90D9")
    ax.fill(angles, baseline_vals, alpha=0.15, color="#4A90D9")
    ax.plot(angles, steered_vals, "s-", linewidth=2, label="Steered (A +2.0)", color="#E85D75")
    ax.fill(angles, steered_vals, alpha=0.15, color="#E85D75")

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=12)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20th", "40th", "60th", "80th", "100th"], size=8, color="grey")
    ax.set_title(title, size=14, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()


radar_chart(baseline_scores, steered_scores)

## Other Inventories

PSYCTL ships with several inventories. Use `create_inventory()` with any of these names:

| Inventory | Items | Measures |
|-----------|-------|----------|
| `ipip_neo_120` | 120 | Big Five (N, E, O, A, C) |
| `ipip_neo_300` | 300 | Big Five (detailed facets) |
| `rei_40` | 40 | Rational-Experiential thinking styles |
| `sd4_28` | 28 | Short Dark Tetrad (Machiavellianism, Narcissism, Psychopathy, Sadism) |
| `vgq_14` | 14 | Victim Gaslighting Questionnaire |
| `indcol_32` | 32 | Individualism-Collectivism |

In [ ]:
# Quick demo: REI-40 (Rational-Experiential Inventory)
rei = create_inventory("rei_40")
rei_questions = rei.get_questions()
print(f"REI-40: {len(rei_questions)} questions, traits: {rei.get_supported_traits()}")

# Quick demo: SD4-28 (Short Dark Tetrad)
sd4 = create_inventory("sd4_28")
sd4_questions = sd4.get_questions()
print(f"SD4-28: {len(sd4_questions)} questions, traits: {sd4.get_supported_traits()}")

print("\nTo score any inventory, just call:")
print('  scores = score_inventory(model, tokenizer, rei.get_questions(), rei)')